# Практическая работа №10. Сравнение моделей классификации изображений и развертывание API

## Цель работы

Провести сравнительный анализ ранее обученных моделей классификации изображений (работы 2-5), выбрать лучшую по метрикам качества, развернуть ее в виде API и создать пользовательский интерфейс.

---
## Раздел 1. Подготовка и анализ моделей

### 1.1. Импорт библиотек

In [ ]:
import os, time, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, Dense, Flatten, Input,
    BatchNormalization, Dropout, GlobalAveragePooling2D
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
from tensorflow.keras.applications import MobileNetV2
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

print('TF version:', tf.__version__)

### 1.2. Загрузка датасета Fashion MNIST

In [ ]:
(trainX, trainy), (testX, testy) = fashion_mnist.load_data()

CLASS_NAMES = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]

x_train = trainX.astype('float32') / 255.0
x_test  = testX.astype('float32')  / 255.0
x_train_cnn = x_train.reshape(-1, 28, 28, 1)
x_test_cnn  = x_test.reshape(-1, 28, 28, 1)

num_classes = 10
y_train = to_categorical(trainy, num_classes)
y_test  = to_categorical(testy, num_classes)

print(f'Train: X = {x_train_cnn.shape}, y = {y_train.shape}')
print(f'Test:  X = {x_test_cnn.shape},  y = {y_test.shape}')
print(f'Классы: {CLASS_NAMES}')

### 1.3. Визуализация примеров из датасета

In [ ]:
plt.figure(figsize=(12, 6))
for i in range(20):
    plt.subplot(2, 10, i + 1)
    plt.imshow(trainX[i], cmap='gray')
    plt.title(CLASS_NAMES[trainy[i]], fontsize=7)
    plt.axis('off')
plt.suptitle('Примеры изображений из Fashion MNIST', fontsize=14)
plt.tight_layout()
plt.show()

### 1.4. Определение и обучение моделей

Сравниваем 4 модели:
1. **Dense NN** (Работа 2) — полносвязная сеть
2. **CNN без BN/Dropout** (Работа 3) — свёрточная сеть
3. **CNN с BN и Dropout** (Работа 4) — свёрточная сеть с регуляризацией
4. **MobileNetV2 Transfer** (Работа 5) — трансферное обучение

In [ ]:
EPOCHS = 20
BATCH_SIZE = 128

def evaluate_model(model, x, y_true_labels, model_name):
    """Оценивает модель: accuracy, precision, recall, F1, время инференса."""
    times = []
    for _ in range(5):
        start = time.time()
        preds = model.predict(x, verbose=0)
        elapsed = time.time() - start
        times.append(elapsed)
    avg_inference_time = np.mean(times)
    avg_per_image_ms = (avg_inference_time / len(x)) * 1000

    y_pred = np.argmax(preds, axis=1)
    acc = accuracy_score(y_true_labels, y_pred)
    prec = precision_score(y_true_labels, y_pred, average='macro', zero_division=0)
    rec = recall_score(y_true_labels, y_pred, average='macro', zero_division=0)
    f1 = f1_score(y_true_labels, y_pred, average='macro', zero_division=0)
    cm = confusion_matrix(y_true_labels, y_pred)

    return {
        'Модель': model_name,
        'Accuracy': round(acc, 4),
        'Precision': round(prec, 4),
        'Recall': round(rec, 4),
        'F1-мера': round(f1, 4),
        'Время инф. (мс/изобр.)': round(avg_per_image_ms, 4),
        'confusion_matrix': cm,
        'y_pred': y_pred
    }

#### Модель 1: Dense NN (Работа 2)

In [ ]:
model_1 = Sequential([
    Input(shape=(28, 28, 1)),
    Flatten(),
    Dense(512, activation='relu'),
    Dense(256, activation='relu'),
    Dense(128, activation='relu'),
    Dense(num_classes, activation='softmax'),
], name='dense_nn')

model_1.compile(optimizer=Adam(learning_rate=1e-3),
                loss='categorical_crossentropy', metrics=['accuracy'])
model_1.summary()

history_1 = model_1.fit(x_train_cnn, y_train,
                        validation_data=(x_test_cnn, y_test),
                        epochs=EPOCHS, batch_size=BATCH_SIZE,
                        callbacks=[ReduceLROnPlateau(patience=3, factor=0.5),
                                   EarlyStopping(patience=6, restore_best_weights=True)],
                        verbose=2)

result_1 = evaluate_model(model_1, x_test_cnn, testy, 'Dense NN (Работа 2)')
print(f"Dense NN: Accuracy={result_1['Accuracy']}, F1={result_1['F1-мера']}")

#### Модель 2: CNN без BN/Dropout (Работа 3)

In [ ]:
model_2 = Sequential([
    Input(shape=(28, 28, 1)),
    Conv2D(32, (3, 3), padding='same', activation='relu'),
    Conv2D(32, (3, 3), padding='same', activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),
    Conv2D(64, (3, 3), padding='same', activation='relu'),
    Conv2D(64, (3, 3), padding='same', activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),
    Conv2D(128, (3, 3), padding='same', activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),
    Flatten(),
    Dense(256, activation='relu'),
    Dense(num_classes, activation='softmax'),
], name='cnn_simple')

model_2.compile(optimizer=Adam(learning_rate=1e-3),
                loss='categorical_crossentropy', metrics=['accuracy'])
model_2.summary()

history_2 = model_2.fit(x_train_cnn, y_train,
                        validation_data=(x_test_cnn, y_test),
                        epochs=EPOCHS, batch_size=BATCH_SIZE,
                        callbacks=[ReduceLROnPlateau(patience=3, factor=0.5),
                                   EarlyStopping(patience=6, restore_best_weights=True)],
                        verbose=2)

result_2 = evaluate_model(model_2, x_test_cnn, testy, 'CNN без BN/Dropout (Работа 3)')
print(f"CNN: Accuracy={result_2['Accuracy']}, F1={result_2['F1-мера']}")

#### Модель 3: CNN с BN и Dropout (Работа 4)

In [ ]:
model_3 = Sequential([
    Input(shape=(28, 28, 1)),
    Conv2D(32, (3, 3), padding='same', activation='relu'), BatchNormalization(),
    Conv2D(32, (3, 3), padding='same', activation='relu'), BatchNormalization(),
    MaxPooling2D((2, 2)), Dropout(0.25),
    Conv2D(64, (3, 3), padding='same', activation='relu'), BatchNormalization(),
    Conv2D(64, (3, 3), padding='same', activation='relu'), BatchNormalization(),
    MaxPooling2D((2, 2)), Dropout(0.25),
    Conv2D(128, (3, 3), padding='same', activation='relu'), BatchNormalization(),
    Conv2D(128, (3, 3), padding='same', activation='relu'), BatchNormalization(),
    MaxPooling2D((2, 2)), Dropout(0.25),
    Flatten(),
    Dense(512, activation='relu'), BatchNormalization(), Dropout(0.5),
    Dense(num_classes, activation='softmax'),
], name='cnn_bn_dropout')

model_3.compile(optimizer=Adam(learning_rate=1e-3),
                loss='categorical_crossentropy', metrics=['accuracy'])
model_3.summary()

history_3 = model_3.fit(x_train_cnn, y_train,
                        validation_data=(x_test_cnn, y_test),
                        epochs=30, batch_size=BATCH_SIZE,
                        callbacks=[ReduceLROnPlateau(patience=3, factor=0.5),
                                   EarlyStopping(patience=8, restore_best_weights=True)],
                        verbose=2)

result_3 = evaluate_model(model_3, x_test_cnn, testy, 'CNN с BN и Dropout (Работа 4)')
print(f"CNN+BN+Dropout: Accuracy={result_3['Accuracy']}, F1={result_3['F1-мера']}")

#### Модель 4: Transfer Learning — MobileNetV2 (Работа 5)

In [ ]:
# Fashion MNIST 28x28 grayscale -> 32x32 RGB для MobileNetV2
x_train_rgb = np.repeat(x_train_cnn, 3, axis=-1)
x_test_rgb  = np.repeat(x_test_cnn, 3, axis=-1)
x_train_rgb = tf.image.resize(x_train_rgb, (32, 32)).numpy()
x_test_rgb  = tf.image.resize(x_test_rgb, (32, 32)).numpy()

base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(32, 32, 3))
base_model.trainable = False

inputs = Input(shape=(32, 32, 3))
x = base_model(inputs, training=False)
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.4)(x)
outputs = Dense(num_classes, activation='softmax')(x)
model_4 = Model(inputs, outputs, name='mobilenetv2_transfer')

model_4.compile(optimizer=Adam(1e-3), loss='categorical_crossentropy', metrics=['accuracy'])
model_4.summary()

# Этап 1: обучение головы
print('Этап 1: обучение классификационной головы...')
model_4.fit(x_train_rgb, y_train, validation_data=(x_test_rgb, y_test),
            epochs=10, batch_size=128, verbose=2)

# Этап 2: fine-tuning
print('Этап 2: fine-tuning последних 20 слоёв...')
base_model.trainable = True
for layer in base_model.layers[:-20]:
    layer.trainable = False
model_4.compile(optimizer=Adam(1e-4), loss='categorical_crossentropy', metrics=['accuracy'])
model_4.fit(x_train_rgb, y_train, validation_data=(x_test_rgb, y_test),
            epochs=10, batch_size=64,
            callbacks=[ReduceLROnPlateau(patience=2, factor=0.5),
                       EarlyStopping(patience=4, restore_best_weights=True)],
            verbose=2)

result_4 = evaluate_model(model_4, x_test_rgb, testy, 'MobileNetV2 Transfer (Работа 5)')
print(f"MobileNetV2: Accuracy={result_4['Accuracy']}, F1={result_4['F1-мера']}")

### 1.5. Сводная таблица результатов

In [1]:
all_results = [result_1, result_2, result_3, result_4]

df_results = pd.DataFrame([{
    'Модель': r['Модель'],
    'Accuracy': r['Accuracy'],
    'Precision': r['Precision'],
    'Recall': r['Recall'],
    'F1-мера': r['F1-мера'],
    'Время инф. (мс/изобр.)': r['Время инф. (мс/изобр.)']
} for r in all_results])

df_results = df_results.set_index('Модель')
print('Сводная таблица результатов сравнения моделей:')
df_results

Сводная таблица результатов сравнения моделей:


,Accuracy,Precision,Recall,F1-мера,Время инф. (мс/изобр.)
Модель,,,,,
Dense NN (Работа 2),0.888,0.8888,0.888,0.8867,0.0272
CNN без BN/Dropout (Работа 3),0.9143,0.9174,0.9143,0.9149,0.0934
CNN с BN и Dropout (Работа 4),0.9058,0.9161,0.9058,0.905,0.1977
MobileNetV2 Transfer (Работа 5),0.8703,0.8715,0.8703,0.8708,0.3771


### Результаты сравнения моделей

| Модель | Accuracy | Precision | Recall | F1-мера | Время инф. (мс/изобр.) |
|--------|----------|-----------|--------|---------|------------------------|
| Dense NN (Работа 2) | 0.888 | 0.8888 | 0.888 | 0.8867 | 0.0272 |
| CNN без BN/Dropout (Работа 3) | 0.9143 | 0.9174 | 0.9143 | 0.9149 | 0.0934 |
| CNN с BN и Dropout (Работа 4) | 0.9058 | 0.9161 | 0.9058 | 0.905 | 0.1977 |
| MobileNetV2 Transfer (Работа 5) | 0.8703 | 0.8715 | 0.8703 | 0.8708 | 0.3771 |


### 1.6. Визуализация результатов сравнения

In [ ]:
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-мера']
model_names = df_results.index.tolist()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
colors = ['#3498db', '#2ecc71', '#e74c3c', '#9b59b6']

for idx, (metric, ax) in enumerate(zip(metrics, axes.flat)):
    values = df_results[metric].values
    bars = ax.barh(model_names, values, color=colors)
    ax.set_xlabel(metric)
    ax.set_title(f'Сравнение по {metric}')
    ax.set_xlim(0, 1)
    for bar, val in zip(bars, values):
        ax.text(val + 0.01, bar.get_y() + bar.get_height() / 2,
                f'{val:.4f}', va='center', fontsize=10)

plt.suptitle('Сравнение моделей по метрикам качества', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

### 1.7. Матрицы ошибок

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(20, 18))

for ax, result in zip(axes.flat, all_results):
    cm = result['confusion_matrix']
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                ax=ax, cbar=False)
    ax.set_title(result['Модель'], fontsize=12, fontweight='bold')
    ax.set_xlabel('Предсказанный класс')
    ax.set_ylabel('Истинный класс')
    ax.tick_params(axis='x', rotation=45)
    ax.tick_params(axis='y', rotation=0)

plt.suptitle('Матрицы ошибок (Confusion Matrices)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

### 1.8. Графики обучения

In [ ]:
histories = [
    ('Dense NN (Работа 2)', history_1),
    ('CNN без BN/Dropout (Работа 3)', history_2),
    ('CNN с BN и Dropout (Работа 4)', history_3),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, hist) in zip(axes.flat, histories):
    ax.plot(hist.history['accuracy'], label='Train Accuracy')
    ax.plot(hist.history['val_accuracy'], label='Val Accuracy')
    ax.plot(hist.history['loss'], '--', label='Train Loss')
    ax.plot(hist.history['val_loss'], '--', label='Val Loss')
    ax.set_title(name, fontsize=11, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Графики обучения моделей', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

### 1.9. Выбор лучшей модели и сохранение

In [2]:
best_idx = df_results['F1-мера'].idxmax()
print(f'Лучшая модель по F1-мере: {best_idx}')
print(f'F1 = {df_results.loc[best_idx, "F1-мера"]}')
print()
print(df_results.sort_values('F1-мера', ascending=False))

model_map = {
    'Dense NN (Работа 2)': model_1,
    'CNN без BN/Dropout (Работа 3)': model_2,
    'CNN с BN и Dropout (Работа 4)': model_3,
    'MobileNetV2 Transfer (Работа 5)': model_4,
}

best_model = model_map[best_idx]
os.makedirs('models', exist_ok=True)
best_model.save('models/best_classification_model.keras')
print(f'\nЛучшая модель сохранена: models/best_classification_model.keras')

Лучшая модель по F1-мере: CNN без BN/Dropout (Работа 3)
F1 = 0.9149

  CNN без BN/Dropout (Работа 3): F1=0.9149, Acc=0.9143
  CNN с BN и Dropout (Работа 4): F1=0.905, Acc=0.9058
  Dense NN (Работа 2): F1=0.8867, Acc=0.888
  MobileNetV2 Transfer (Работа 5): F1=0.8708, Acc=0.8703

Лучшая модель сохранена: models/best_classification_model.keras


---
## Раздел 2. Развертывание API

Создано FastAPI-приложение (`api/main.py`) со следующими возможностями:
1. Загрузка лучшей модели при старте
2. Эндпоинт `POST /predict` — принимает изображение, возвращает класс и вероятности
3. Предобработка: конвертация в grayscale, resize до 28×28, нормализация [0, 1]
4. Автоматическая инверсия изображений с светлым фоном
5. CORS-поддержка

### Запуск
```bash
cd api
pip install -r requirements.txt
uvicorn main:app --host 0.0.0.0 --port 8000
```

### Пример запроса
```bash
curl -X POST "http://localhost:8000/predict" -F "file=@test_image.png"
```

### Пример ответа
```json
{
    "predicted_class": "Ankle boot",
    "predicted_class_index": 9,
    "confidence": 0.9871,
    "probabilities": { ... }
}
```

#### Ответ:

**ВАША ССЫЛКА НА БЭКЕНД (Render.com)**

---
## Раздел 3. Streamlit-интерфейс

Создано Streamlit-приложение (`streamlit_app/app.py`):
1. Загрузка изображения (png, jpg, bmp, webp)
2. Рисование на холсте (белым по чёрному фону)
3. Предобработка и отправка на API
4. Отображение класса и уверенности
5. Визуализация вероятностей (Plotly)

### Запуск
```bash
cd streamlit_app
pip install -r requirements.txt
streamlit run app.py
```

#### Ответ:

**ВАША ССЫЛКА НА ФРОНТЭНД (Streamlit Cloud)**

---
## Раздел 4. Структура проекта

```
fashion-mnist-classification/
├── api/
│   ├── main.py                          # FastAPI-приложение
│   └── requirements.txt
├── streamlit_app/
│   ├── app.py                           # Streamlit-приложение
│   └── requirements.txt
├── models/
│   └── best_classification_model.keras  # Лучшая модель
├── practical_work_10.ipynb              # Данный ноутбук
├── train_models.py                      # Скрипт обучения
├── Dockerfile                           # Для Render.com
├── README.md
└── .gitignore
```

#### Ответ:

**ВАША ССЫЛКА НА РЕПОЗИТОРИЙ**